In [21]:
import warnings; warnings.filterwarnings(action='ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from binance import Client

In [22]:
client = Client()
def get_tickers():
    tickers = []
    for info in client.futures_mark_price():
        ticker = info['symbol']
        if ticker[-4:] == "USDT":
            tickers.append(info['symbol'])
    return tickers

def get_sample(ticker, start_date="1 Jan, 2021"):
    klines = np.array(client.futures_historical_klines(ticker, Client.KLINE_INTERVAL_1DAY, start_date))
    sample = pd.DataFrame(klines.reshape(-1, 12), dtype=float, columns=['datetime',
                                                                        'open', 
                                                                        'high', 
                                                                        'low', 
                                                                        'close', 
                                                                        'volume', 
                                                                        'close time', 
                                                                        'quote asset volume, number of trades', 
                                                                        'number of trades',
                                                                        'taker buy base asset volume', 
                                                                        'taker buy quote asset volume', 
                                                                        'ignore'])
    sample['datetime'] = pd.to_datetime(sample['datetime'], unit='ms')
    sample.set_index('datetime', inplace=True)
    sample = sample[['open', 'high', 'low', 'close', 'volume']].copy()
    return sample

In [23]:
tickers = get_tickers()

# drop coins with length < 200
for ticker in tickers:
    sample = get_sample(ticker, start_date="1 Mar, 2021")
    if len(sample) < 200: tickers.remove(ticker)
    else: continue

In [24]:
# define preprocessing functions
def add_momentum(sample, period=20):
    df = sample.copy()
    df[f'mom{period}'] = (df['close'] - df.shift(period)['close'])/df.shift(period)['close']
    sample[f'mom{period}'] = df[f'mom{period}']; del df
    return sample

def add_noise(sample, period=20):
    df = sample.copy()
    df['noise'] = 1. - abs(df['close'] - df['open'])/(df['high'] - df['low'])
    df[f'noise{period}'] = df['noise'].rolling(period).mean()
    sample[f'noise{period}'] = df[f"noise{period}"]; del df
    return sample

def add_percentB(sample, period=20):
    df = sample.copy()
    df['center'] = df['close'].rolling(period).mean()
    df['upper'] = df['center'] + 2.*df['close'].rolling(period).std()
    df['lower'] = df['center'] - 2.*df['close'].rolling(period).std()
    df['percentB'] = (df['close'] - df['lower'])/(df['upper'] - df['lower'])
    sample['percentB'] = df['percentB']; del df
    return sample

def add_RSI(sample, period=14):
    df = sample.copy()
    df['TP'] = (df['high'] + df['low'] + df['close'])/3.
    df['U'] = 0.; df['D'] = 0.
    for idx in df.index:
        if df.shift(1).loc[idx, 'TP'] < df.loc[idx, 'TP']:
            df.loc[idx, 'U'] = df.loc[idx, 'TP']
        else:
            df.loc[idx, 'D'] = df.loc[idx, 'TP']
    df['AU'] = df['U'].rolling(period).mean()
    df['AD'] = df['D'].rolling(period).mean()
    df['RSI'] = df['AU']/(df['AU']+df['AD'])
    sample['TP'] = df['TP']
    sample['RSI'] = df['RSI']; del df
    return sample

In [25]:
# Test will be based on 14 days momentum / PB / RSI
samples = dict()
for ticker in tickers:
    sample = get_sample(ticker, start_date="1 Apr, 2020")   # at least 21 coins are possible
    
    # preprocess
    sample = add_momentum(sample, period=20)
    sample = add_percentB(sample)
    sample = add_RSI(sample)
    sample['reward'] = 1. + sample['close'].pct_change()
    sample.dropna(inplace=True)
    
    samples[ticker] = sample.copy(); del sample
reference = samples['BTCUSDT']

In [26]:
# 1. choose top 21 trading coins
# 2. sort in indicator order
# 3. compare top 5 / bottom 5 momentum coins
def inspect_indicator(indicator):
    top5 = samples['BTCUSDT'][['close']].copy()
    top5['number'] = top5.index.map(mdates.date2num)
    top5['reward'] = 1. 
    bottom5 = top5.copy()

    for idx in top5.index[:-1]:
        # select 21 highest volume coins
        volumes = dict()
        for ticker in tickers:
            try:
                volumes[ticker] = samples[ticker].loc[idx, 'TP']*samples[ticker].loc[idx, 'volume']
            except:
                continue
        top21v = dict(sorted(volumes.items(), key=(lambda x: x[1]), reverse=True)[:21])
    
        # sort in indicator order
        temp = dict()
        for ticker in top21v.keys():
            try:
                temp[ticker] = samples[ticker].loc[idx, indicator]
            except:
                print(ticker)
        temp = dict(sorted(temp.items(), key=(lambda x: x[1]), reverse=True))
    
        # estimate reward for top/bottom 5
        reward_top = 0.; reward_bottom = 0.
        for coin in list(temp.keys())[:5]:
            this_reward = samples[coin].shift(-1).loc[idx, 'reward'] - 0.003
            reward_top += 0.2*this_reward
        for coin in list(temp.keys())[-5:]:
            this_reward = samples[coin].shift(-1).loc[idx, 'reward'] - 0.003
            reward_bottom += 0.2*this_reward
    
        top5.loc[idx, 'reward'] = reward_top
        bottom5.loc[idx, 'reward'] = reward_bottom
        
    total_reward = 1.
    for idx in top5.index:
        total_reward *= top5.loc[idx, 'reward']
        top5.loc[idx, 'total_reward'] = total_reward
    total_reward = 1.
    for idx in bottom5.index:
        total_reward *= bottom5.loc[idx, 'reward']
        bottom5.loc[idx, 'total_reward'] = total_reward
    
    return (top5, bottom5)

In [27]:
def evaluate(book, title=""):
    # CAGR, MDD, Volatility, Sharpe
    CAGR = book['total_reward'].iloc[-1]**(365/len(book.index)) - 1.

    historical_max = book['total_reward'].cummax()
    daily_drawdown = book['total_reward']/historical_max - 1.
    historical_dd = daily_drawdown.cummin()
    MDD = historical_dd.min()
    VOL = np.std(book['reward'])*np.sqrt(365.)
    Sharpe = (np.mean(book['reward'])/np.std(book['reward'])*np.sqrt(365.))

    # win-loose ratio
    win = 0; loose = 0
    for idx in book.index:
        if book.loc[idx, 'reward'] > 1.:
            win += 1
        else:
            loose += 1
    win_loose_ratio = win/(win+loose)

    print(f"==== {title} ====")
    print(f"Accumulated Returns: {(total_reward-1.)*100:.2f}%")
    print(f"CAGR: {CAGR*100:.2f}%")
    print(f"MDD: {MDD*100:.2f}%")
    print(f"VOL: {VOL*100:.2f}%")
    print(f"Sharpe: {Sharpe*100:.2f}%")
    print(f"win-loose ratio: {win_loose_ratio*100:.2f}%")
    print()

In [28]:
top5, bottom5 = inspect_indicator("mom20")
evaluate(top5, "momentum 20 days - top5")
evaluate(bottom5, "momentum 20 days - bottom5")

top5, bottom5 = inspect_indicator("percentB")
evaluate(top5, "Percent B - top5")
evaluate(bottom5, "Percnet B - bottom5")

top5, bottom5 = inspect_indicator("RSI")
evaluate(top5, "RSI - top5")
evaluate(bottom5, "RSI - bottom5")


==== momentum 20 days - top5 ====
Accumulated Returns: 5773.90%
CAGR: 731.27%
MDD: -70.54%
VOL: 134.51%
Sharpe: 27359.95%
win-loose ratio: 54.27%

==== momentum 20 days - bottom5 ====
Accumulated Returns: -70.16%
CAGR: -46.67%
MDD: -86.64%
VOL: 110.55%
Sharpe: 33015.75%
win-loose ratio: 51.42%

==== Percent B - top5 ====
Accumulated Returns: 1934.05%
CAGR: 378.93%
MDD: -77.36%
VOL: 132.67%
Sharpe: 27694.41%
win-loose ratio: 53.56%

==== Percnet B - bottom5 ====
Accumulated Returns: -91.16%
CAGR: -71.67%
MDD: -93.45%
VOL: 104.02%
Sharpe: 35022.40%
win-loose ratio: 50.14%

==== RSI - top5 ====
Accumulated Returns: 2505.24%
CAGR: 444.70%
MDD: -73.88%
VOL: 127.06%
Sharpe: 28922.25%
win-loose ratio: 52.99%

==== RSI - bottom5 ====
Accumulated Returns: -35.14%
CAGR: -20.16%
MDD: -79.20%
VOL: 111.18%
Sharpe: 32864.16%
win-loose ratio: 50.71%



### Conclusion
- Among momentum, RSI, PercentB, momentum is the best indicator